# Inventory Policy And Slotting Readiness

Shows empirical lead-time `(s,S)` decisions, service classes, handling-unit rounding and physical space requirements used by the optimizer.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("Ai miroservices/modeling/project_operational_baseline").resolve()
OUT = ROOT / "outputs"
EVAL = OUT / "evaluator"
sns.set_theme(style="whitegrid")

In [ ]:
policy=pd.read_csv(OUT/'inventory_policy.csv.gz')
materials=pd.read_csv(OUT/'materials.csv.gz')
locations=pd.read_csv(OUT/'locations.csv.gz')
display(policy.groupby('amalgamated_class')[['target_service_level','safety_stock','required_handling_units']].agg(['mean','median','sum']))

In [ ]:
assert (policy.max_stock>=policy.min_stock).all()
assert (policy.order_quantity%policy.units_per_handling_unit==0).all()
print('Policy feasibility checks passed.')
print('Required handling units:',policy.required_handling_units.sum(),'Available pallet-capacity units:',locations.max_pallet_capacity.sum())

In [ ]:
print('Required pallet positions:',policy.required_pallet_positions.sum(),'Available pallet positions:',locations.max_pallet_capacity.sum())
sns.scatterplot(data=policy,x='expected_holding_cost',y='expected_shortage_cost',hue='amalgamated_class',alpha=.55); plt.xscale('symlog'); plt.yscale('symlog'); plt.show()

In [ ]:
cost=pd.read_csv(EVAL/'decision_cost_sensitivity.csv')
primary_cost=cost[cost.population.eq('RM_PM_PRIMARY')]
display(primary_cost.sort_values(['under_to_over_cost_ratio','weighted_total_cost_proxy']))
display(cost[cost.population.eq('FG_SECONDARY')].sort_values(['under_to_over_cost_ratio','weighted_total_cost_proxy']))
sns.catplot(data=primary_cost,x='weighted_total_cost_proxy',y='model_name',col='under_to_over_cost_ratio',kind='bar',col_wrap=2,sharex=False,height=4); plt.show()

The 1:1, 2:1, 3:1 and 5:1 results are sensitivity evidence. No cost ratio is promoted to a production objective until shortage, holding and service costs are supplied and approved.